# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sumit-M-Poonia/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

###  ML Task Type
* **Chosen Lane:** Lane 1 — Content Refresh Prioritization
* **ML Task Type:** Binary Classification with Probability Scoring / Ranking
* **Core Objective:** Predict whether a given web page will experience a sustained search traffic decline ($1 = \text{Declining}$, $0 = \text{Stable/Growing}$), and rank all client URLs by their predicted risk probability score $P(\text{declining})$.
* **Action Supported:** SEO and content teams use the top-$K$ ranked list to allocate editorial budgets toward updating high-risk pages before traffic drops fully materialize.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

###  Target Variable & Proxy Definition
* **Ideal Ground Truth Target:** The percentage drop in organic search sessions over the next 90-day forward window ($\%\Delta \text{Traffic}_{t+90} < -15\%$).
* **Practical Proxy Target:** Historical 90-day trend direction from search performance metrics (`trend_direction == 'down'`).
* **Target Binary Column:** `target_is_declining = 1` if `trend_direction == 'down'`, else `0`.
* **Proxy Validation:** Since future traffic requires time-lagged observation, historical multi-metric trend classification serves as a reliable, immediately available proxy label for model training.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

###  Business-Aligned Success Metric
* **Primary Evaluation Metric:** Precision@K (e.g., Precision@50 or Precision@100)
* **Secondary Evaluation Metric:** PR-AUC (Precision-Recall Area Under Curve)

**Why Precision@K fits the business action:**
* Content teams operate under fixed capacity constraints (e.g., budget to refresh only 50 articles per month).
* A False Positive (flagging a healthy page) wastes $200–$500 in writer fees and risks damaging a top-ranking URL.
* Precision@50 measures the percentage of actually declining pages within the top 50 model recommendations. Maximizing Precision@K ensures that every dollar spent on a refresh goes toward a page that actually needed intervention.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
import os
import pandas as pd
import numpy as np

# Fallback URL for Colab standalone runs
raw_github_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

possible_paths = [
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv"
]

data_source = next((p for p in possible_paths if os.path.exists(p)), raw_github_url)
df = pd.read_csv(data_source)

# Create binary proxy target
df["target_is_declining"] = (df["trend_direction"] == "down").astype(int)

# Identify features and target column
feature_cols = [
    "content_age_days", "days_since_last_update",
    "impressions_90d", "avg_position", "ctr", "word_count"
]
available_features = [c for c in feature_cols if c in df.columns]

# Display summary of unit of analysis
print("=== UNIT OF ANALYSIS ===")
print(f"Unit of Analysis: 1 Row = 1 Web Page / URL")
print(f"Total Rows (Pages): {len(df):,}")
print(f"Target Variable Distribution (target_is_declining):")
print(df["target_is_declining"].value_counts(normalize=True).round(3).to_string())
print("\nSample Dataframe Representation:")

# Show dataframe preview with features and created target
df[["target_is_declining"] + available_features].head()

=== UNIT OF ANALYSIS ===
Unit of Analysis: 1 Row = 1 Web Page / URL
Total Rows (Pages): 30,000
Target Variable Distribution (target_is_declining):
target_is_declining
1    0.542
0    0.458

Sample Dataframe Representation:


,target_is_declining,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count
0,1,187,20,3803,10.6,0.76,3221.0
1,1,445,25,15320,20.3,0.05,2481.0
2,1,141,20,12581,36.5,0.09,3515.0
3,0,463,22,11751,6.2,0.49,NaN
4,1,263,14,19140,44.0,0.13,2803.0


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

###  Why ML Beats Static Rules
* **Rule-Based Failure Mode (High False Positives):** A naive rule like "Flag all content older than 365 days" fails because authoritative evergreen content often stays ranked #1 for years. A static age rule wastes editorial budget on top-performing legacy pages.
* **Multi-Signal Interaction:** Content decay is non-linear. A drop in CTR paired with declining search position and low word count signals competitive decay much stronger than any single variable in isolation. Machine learning captures these high-dimensional feature interactions automatically.
* **Ranked Prioritization vs. Binary Cutoffs:** Fixed rules produce unordered lists of flagged pages. ML outputs calibrated probabilities $P(\text{declining})$, allowing teams to dynamically adjust cutoff $K$ to match available monthly writer capacity.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check
- [x] **Task Type:** Explicitly defined as Binary Classification / Risk Ranking.
- [x] **Target/Proxy:** Ground truth binary proxy (`target_is_declining`) created from trend direction.
- [x] **Success Metric:** Precision@K selected to align with fixed editorial refresh budgets.
- [x] **Unit of Analysis:** Verified via code cell as 1 row = 1 URL/page.
- [x] **Value Over Heuristics:** Explained multi-signal interactions and probability calibration over rigid rules.